# **Feature Engineering Notebook**

## Objectives

* Encode categorical variables for use in ML pipelines.
* Apply ordinal encoding to ordered features (`experience_level`, `company_size`, `education_required`).
* Apply frequency encoding to high-cardinality columns (`company_location`, `employee_residence`).
* Assess transformed features and their relationship with `salary_usd`.

## Inputs

* `outputs/datasets/cleaned/ai_jobs_cleaned.csv`

## Outputs

* `outputs/datasets/featured/ai_jobs_featured.csv`
* Insights that inform the pipeline steps in notebook 05 and 06.

---

# Change working directory

In [2]:
import os
current_dir = os.getcwd()
os.chdir(os.path.dirname(current_dir))
print("Working directory:", os.getcwd())

Working directory: c:\Users\chahi\Desktop\vscode-project\the-ai-salary-index


# Load Data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from feature_engine.encoding import OrdinalEncoder
from sklearn.preprocessing import OrdinalEncoder 

%matplotlib inline

TrainSet = pd.read_csv("outputs/datasets/cleaned/TrainSet.csv")
TestSet = pd.read_csv("outputs/datasets/cleaned/TestSet.csv")
print(f"TrainSet Shape: {TrainSet.shape}")
print(f"TestSet Shape: {TestSet.shape}")
TrainSet.head()

TrainSet Shape: (11760, 14)
TestSet Shape: (2941, 14)


,job_title,salary_usd,experience_level,employment_type,company_location,company_size,employee_residence,remote_ratio,required_skills,education_required,years_experience,industry,benefits_score,company_name
0,Machine Learning Engineer,177700,EX,PT,United Kingdom,M,United Kingdom,100,"Kubernetes, SQL, Python",PhD,14,Retail,8.2,Quantum Computing Inc
1,Principal Data Scientist,226559,EX,PT,South Korea,L,Argentina,50,"R, Python, Azure, SQL",PhD,11,Consulting,8.0,Future Systems
2,Research Scientist,109363,SE,CT,Singapore,S,Singapore,100,"Deep Learning, Java, Python",Associate,6,Energy,7.9,Predictive Systems
3,Data Analyst,130819,EX,FT,Finland,M,Finland,100,"Java, R, Mathematics",PhD,15,Telecommunications,8.5,Neural Networks Co
4,NLP Engineer,66333,EN,FL,Australia,M,Australia,100,"SQL, Kubernetes, Hadoop",Bachelor,1,Retail,6.8,Algorithmic Solutions


# Feature Engineering

## Ordinal Encoding — experience_level

In [ ]:
exp_map = {'EN': 0, 'MI': 1, 'SE': 2, 'EX': 3}
TrainSet['experience_level_enc'] = TrainSet['experience_level'].map(exp_map)
TestSet['experience_level_enc'] = TestSet['experience_level'].map(exp_map)
print(TrainSet[['experience_level', 'experience_level_enc']].drop_duplicates().sort_values('experience_level_enc'))

## Ordinal Encoding — education_required

## Ordinal Encoding — company_size

In [ ]:
size_map = {'S': 0, 'M': 1, 'L': 2}
TrainSet['company_size_enc'] = TrainSet['company_size'].map(size_map)
TestSet['company_size_enc'] = TestSet['company_size'].map(size_map)
print(TrainSet[['company_size', 'company_size_enc']].drop_duplicates().sort_values('company_size_enc'))

In [ ]:
edu_map = {
    'Associate': 0,
    'Bachelor': 1,
    'Master': 2,
    'PhD': 3
}
TrainSet['education_required_enc'] = TrainSet['education_required'].map(edu_map)
TestSet['education_required_enc'] = TestSet['education_required'].map(edu_map)
print("NaNs in TrainSet:", TrainSet['education_required_enc'].isna().sum())
print("NaNs in TestSet:", TestSet['education_required_enc'].isna().sum())
print(TrainSet[['education_required', 'education_required_enc']].drop_duplicates().sort_values('education_required_enc'))

## Ordinal Encoding — company_size

In [ ]:
size_map = {'S': 0, 'M': 1, 'L': 2}
TrainSet['company_size_enc'] = TrainSet['company_size'].map(size_map)
TestSet['company_size_enc'] = TestSet['company_size'].map(size_map)
print(TrainSet[['company_size', 'company_size_enc']].drop_duplicates())

## One-Hot Encoding — employment_type

In [ ]:
TrainSet = pd.get_dummies(TrainSet, columns=['employment_type'], drop_first=True, dtype=int)
TestSet = pd.get_dummies(TestSet, columns=['employment_type'], drop_first=True, dtype=int)

# Align TestSet columns to TrainSet in case of missing categories
TestSet = TestSet.reindex(columns=TrainSet.columns, fill_value=0)

print(TrainSet.filter(like='employment_type').head())

## Frequency Encoding — company_location & employee_residence

In [ ]:
for col in ['company_location', 'employee_residence']:
    # Compute frequency map from TrainSet only — apply same map to TestSet
    freq_map = TrainSet[col].value_counts(normalize=True)
    TrainSet[f'{col}_freq'] = TrainSet[col].map(freq_map)
    TestSet[f'{col}_freq'] = TestSet[col].map(freq_map)
    print(f"{col}_freq sample:")
    print(TrainSet[[col, f'{col}_freq']].drop_duplicates().head(5))
    print()

## Encoding education_required

In [ ]:
print("Unique values:", TrainSet['education_required'].unique())

edu_map = {
    'Associate': 0,
    'Bachelor': 1,
    'Master': 2,
    'PhD': 3
}

TrainSet['education_required_enc'] = TrainSet['education_required'].map(edu_map)
TestSet['education_required_enc'] = TestSet['education_required'].map(edu_map)

# Verify no NaNs — map should cover all unique values
print("NaNs in TrainSet:", TrainSet['education_required_enc'].isna().sum())
print("NaNs in TestSet:", TestSet['education_required_enc'].isna().sum())

print(TrainSet[['education_required', 'education_required_enc']].drop_duplicates().sort_values('education_required_enc'))

## Assess Transformed Features

In [ ]:
num_cols = TrainSet.select_dtypes(include='number').columns.tolist()
corr = TrainSet[num_cols].corr()['salary_usd'].drop('salary_usd').sort_values(key=abs, ascending=False)
print("Correlation with salary_usd (post-feature engineering):")
print(corr.head(15))

# Save Featured Dataset

In [ ]:
os.makedirs('outputs/datasets/featured', exist_ok=True)
TrainSet.to_csv('outputs/datasets/featured/ai_jobs_featured_train.csv', index=False)
TestSet.to_csv('outputs/datasets/featured/ai_jobs_featured_test.csv', index=False)
print(f"Saved featured dataset. Shape: {TrainSet.shape}")
TrainSet.head()

---

# Conclusions and Next Steps

* Ordinal encoding applied to `experience_level`, `company_size`, `education_required`.
* One-hot encoding applied to `employment_type`.
* Frequency encoding applied to `company_location` and `employee_residence`.
* Feature-engineered dataset saved.

Next step: **05 - ModelingEvaluation - PredictSalary** — build and evaluate a regression model.